# Notebook-template — Spark + Delta + MinIO

Base parametrizável (Papermill) para os notebooks das camadas do Data Lake
(Landing -> Bronze -> Silver -> Gold). Issue #24 / épico #15.

In [ ]:
import os

# Parâmetros sobrescrevíveis pelo Papermill (-p nome valor)
layer = "template"
dataset = "exemplo"
run_date = "2026-06-17"
minio_endpoint = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.environ.get("MINIO_ROOT_USER", "minioadmin")
minio_secret_key = os.environ.get("MINIO_ROOT_PASSWORD", "minioadmin")
bucket = os.environ.get("DATALAKE_BUCKET", "datalake")

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName(f"engdb-{layer}-{dataset}")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)
spark = configure_spark_with_delta_pip(
    builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]
).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| destino bucket:", bucket)

In [ ]:
# Exemplo end-to-end: grava e lê uma tabela Delta no MinIO (valida S3A + Delta)
path = f"s3a://{bucket}/{layer}/{dataset}"

df = spark.createDataFrame(
    [(1, "a", run_date), (2, "b", run_date)],
    ["id", "valor", "run_date"],
)
df.write.format("delta").mode("overwrite").save(path)

back = spark.read.format("delta").load(path)
print("linhas lidas de volta:", back.count())
back.show()

spark.stop()